# Training and inference energy measurements

Run this after Notebook 04. It measures the same tabular models on validated and significant data and saves each run under its matching Results subfolder. Actual joules require Linux RAPL or a local Scaphandre counter; Windows without Scaphandre will report measurements unavailable. CodeCarbon outputs, when enabled, are estimates only.


In [ ]:
import json
import re
import sys
import time
from pathlib import Path
from urllib.request import urlopen

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "Data" / "Processed").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run Notebook 03 and Notebook 04 first.")
PROCESSED_DIR = ROOT / "Data" / "Processed"
RESULTS_DIR = ROOT / "Results"
DATASET_NAMES = ("validated", "significant")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
REPETITIONS = 10
RUN_CODECARBON_ESTIMATES = False
SCAPHANDRE_METRICS_URL = "http://localhost:8080/metrics"

dataset_runs = {}
model_names = ["logistic_regression", "random_forest", "xgboost"]
for dataset_name in DATASET_NAMES:
    dataset_file = (
        "prepared_training_dataset.parquet" if dataset_name == "validated"
        else "prepared_significant_training_dataset.parquet"
    )
    model_dir = PROCESSED_DIR / "models" / dataset_name
    frame = pd.read_parquet(PROCESSED_DIR / dataset_file)
    metadata_path = model_dir / "training_metadata.json"
    if not metadata_path.is_file():
        raise FileNotFoundError(f"Run Notebook 04 first; missing {metadata_path}")
    model_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    feature_columns = model_metadata["feature_columns"]
    models = {name: joblib.load(model_dir / f"{name}.joblib") for name in model_names}
    X = frame[feature_columns].astype(float)
    y = frame["energy_winner"].astype(int)
    sample_size = min(256, len(X))
    dataset_runs[dataset_name] = {
        "X": X, "y": y, "inference_X": X.sample(n=sample_size, random_state=42),
        "models": models, "sample_size": sample_size,
    }

def find_rapl_counters():
    if not sys.platform.startswith("linux"):
        return []
    counters = []
    for energy_file in Path("/sys/class/powercap").glob("intel-rapl:*/energy_uj"):
        if not re.fullmatch(r"intel-rapl:\\d+", energy_file.parent.name):
            continue
        range_file = energy_file.parent / "max_energy_range_uj"
        if range_file.is_file():
            try:
                int(energy_file.read_text().strip())
                counters.append((energy_file, int(range_file.read_text().strip())))
            except (OSError, ValueError):
                continue
    return counters

def read_rapl_joules(counters):
    return sum(int(energy_file.read_text().strip()) for energy_file, _ in counters) / 1_000_000

def rapl_delta_joules(start_joules, counters):
    current = [int(energy_file.read_text().strip()) for energy_file, _ in counters]
    return sum(
        ((value / 1_000_000) - (start_joules[index] / 1_000_000)) % (limit / 1_000_000)
        for index, (value, (_, limit)) in enumerate(zip(current, counters))
    )

def read_scaphandre_joules(url=SCAPHANDRE_METRICS_URL):
    with urlopen(url, timeout=1) as response:
        metrics = response.read().decode("utf-8")
    for line in metrics.splitlines():
        if line.startswith("scaph_host_energy_microjoules_total"):
            value = float(line.rsplit(None, 1)[-1])
            return value / 1_000_000
    raise ValueError("Scaphandre endpoint has no scaph_host_energy_microjoules_total counter.")

rapl_counters = find_rapl_counters()
scaphandre_available = False
try:
    read_scaphandre_joules()
    scaphandre_available = True
except (OSError, ValueError, TimeoutError):
    pass

if rapl_counters:
    energy_backend = "RAPL"

    def read_energy():
        return read_rapl_joules(rapl_counters)

    def delta_energy(start):
        current = [int(energy_file.read_text().strip()) for energy_file, _ in rapl_counters]
        return sum(
            ((value / 1_000_000) - (start[index] / 1_000_000)) % (limit / 1_000_000)
            for index, (value, (_, limit)) in enumerate(zip(current, rapl_counters))
        )

    def take_counter_snapshot():
        return [int(energy_file.read_text().strip()) for energy_file, _ in rapl_counters]
elif scaphandre_available:
    energy_backend = "Scaphandre host counter"
    take_counter_snapshot = read_scaphandre_joules
    delta_energy = lambda start: read_scaphandre_joules() - start
else:
    energy_backend = None

def measure_with_codecarbon_estimate(workload):
    """Optional CodeCarbon estimate; never label this return value as a sensor measurement."""
    from codecarbon import EmissionsTracker

    tracker = EmissionsTracker(
        project_name="energy_aware_recommendation",
        save_to_file=False,
        save_to_logger=False,
        save_to_api=False,
        log_level="error",
    )
    tracker.start()
    try:
        workload()
    finally:
        tracker.stop()
    energy_kwh = tracker.final_emissions_data.energy_consumed
    return {
        "energy_joules_estimate": float(energy_kwh) * 3_600_000,
        "actual_measurement": False,
        "method": "CodeCarbon estimate (not a hardware-verified measurement)",
    }

print(f"Measurement backend: {energy_backend or 'none'}")
print(f"CodeCarbon available for explicitly labelled estimates: {__import__('importlib').util.find_spec('codecarbon') is not None}")


In [ ]:
def measure_one_repetition(dataset_name, model_name, task, task_name, repetition, sample_size):
    counter_start = take_counter_snapshot()
    started = time.perf_counter()
    task()
    elapsed = time.perf_counter() - started
    gross_joules = delta_energy(counter_start)
    baseline_start = take_counter_snapshot()
    time.sleep(elapsed)
    baseline_joules = delta_energy(baseline_start)
    return {
        "dataset": dataset_name, "model": model_name, "task": task_name,
        "repetition": repetition, "backend": energy_backend, "elapsed_seconds": elapsed,
        "gross_energy_joules": gross_joules, "idle_baseline_joules": baseline_joules,
        "net_energy_joules": gross_joules - baseline_joules,
        "n_inference_pairs": sample_size if task_name == "inference" else np.nan,
        "actual_measurement": True,
    }

measurement_rows = []
status_by_dataset = {}
if energy_backend is None:
    for dataset_name in DATASET_NAMES:
        status_by_dataset[dataset_name] = {
            "dataset": dataset_name, "actual_measurements_available": False,
            "status": "unavailable",
            "reason": "No RAPL counter or local Scaphandre host energy counter was detected.",
            "platform": sys.platform, "requested_repetitions": REPETITIONS,
            "completed_repetitions_per_model_task": 0, "completed_measurements": 0,
            "codecarbon_note": "CodeCarbon estimates are not actual sensor measurements.",
        }
    print("No supported hardware energy counter was detected; actual measurements are unavailable.")
else:
    for dataset_name, run in dataset_runs.items():
        for model_name, model in run["models"].items():
            def train_once(model=model, X=run["X"], y=run["y"]):
                clone(model).fit(X, y)
            def infer_once(model=model, inference_X=run["inference_X"]):
                model.predict(inference_X)
            for task_name, task in (("training", train_once), ("inference", infer_once)):
                for repetition in range(1, REPETITIONS + 1):
                    measurement_rows.append(measure_one_repetition(
                        dataset_name, model_name, task, task_name, repetition, run["sample_size"]
                    ))
        status_by_dataset[dataset_name] = {
            "dataset": dataset_name, "actual_measurements_available": True,
            "status": "measured", "backend": energy_backend,
            "requested_repetitions": REPETITIONS,
            "completed_repetitions_per_model_task": REPETITIONS,
            "completed_measurements": sum(row["dataset"] == dataset_name for row in measurement_rows),
            "platform": sys.platform,
            "baseline": "Same-duration idle interval subtracted; gross and net values retained.",
        }

estimate_rows = []
if RUN_CODECARBON_ESTIMATES:
    for dataset_name, run in dataset_runs.items():
        for model_name, model in run["models"].items():
            tasks = (
                ("training", lambda model=model, X=run["X"], y=run["y"]: clone(model).fit(X, y)),
                ("inference", lambda model=model, X=run["inference_X"]: model.predict(X)),
            )
            for task_name, task in tasks:
                for repetition in range(1, REPETITIONS + 1):
                    estimate_rows.append({
                        "dataset": dataset_name, "model": model_name, "task": task_name,
                        "repetition": repetition, **measure_with_codecarbon_estimate(task),
                    })

measurement_columns = [
    "dataset", "model", "task", "repetition", "backend", "elapsed_seconds",
    "gross_energy_joules", "idle_baseline_joules", "net_energy_joules",
    "n_inference_pairs", "actual_measurement",
]
measurements = pd.DataFrame(measurement_rows, columns=measurement_columns)
for dataset_name in DATASET_NAMES:
    output_dir = RESULTS_DIR / dataset_name
    output_dir.mkdir(parents=True, exist_ok=True)
    dataset_measurements = measurements.loc[measurements["dataset"].eq(dataset_name)]
    dataset_measurements.to_csv(output_dir / "energy_measurements.csv", index=False)
    summary = dataset_measurements.groupby(["model", "task", "backend"], as_index=False).agg(
        repetitions=("net_energy_joules", "count"),
        mean_net_energy_joules=("net_energy_joules", "mean"),
        std_net_energy_joules=("net_energy_joules", "std"),
        mean_gross_energy_joules=("gross_energy_joules", "mean"),
        mean_idle_baseline_joules=("idle_baseline_joules", "mean"),
        inference_pairs=("n_inference_pairs", "mean"),
    )
    summary.to_csv(output_dir / "energy_measurement_summary.csv", index=False)
    (output_dir / "energy_measurement_status.json").write_text(
        json.dumps(status_by_dataset[dataset_name], indent=2), encoding="utf-8"
    )
    pd.DataFrame([row for row in estimate_rows if row["dataset"] == dataset_name]).to_csv(
        output_dir / "energy_estimates_codecarbon.csv", index=False
    )
    if not summary.empty:
        display(summary)
    print(json.dumps(status_by_dataset[dataset_name], indent=2))


### Optional CodeBERT energy measurements

Set `RUN_CODEBERT_MEASUREMENTS = True` to measure CodeBERT training and inference on both data sets. It requires the models from Notebook 04 and a supported RAPL/Scaphandre energy counter. Training includes code embedding extraction; inference is measured on up to 256 pairs ten times. Model download/loading is excluded. Each data set is saved in its own Results folder.


In [ ]:
import sys
import gc
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from experiment_utils import codebert_measurement_tasks

RUN_CODEBERT_MEASUREMENTS = False
CODEBERT_TRAINING_REPETITIONS = 1
CODEBERT_INFERENCE_REPETITIONS = 10
codebert_rows = []
codebert_status = {}

if RUN_CODEBERT_MEASUREMENTS and energy_backend is not None:
    for dataset_name in DATASET_NAMES:
        dataset_file = (
            "prepared_training_dataset.parquet" if dataset_name == "validated"
            else "prepared_significant_training_dataset.parquet"
        )
        tasks = codebert_measurement_tasks(
            PROCESSED_DIR / dataset_file,
            PROCESSED_DIR / "models" / dataset_name,
        )
        for task_name, repetitions in (
            ("training", CODEBERT_TRAINING_REPETITIONS),
            ("inference", CODEBERT_INFERENCE_REPETITIONS),
        ):
            for repetition in range(1, repetitions + 1):
                row = measure_one_repetition(
                    dataset_name,
                    "codebert_frozen_embeddings_logistic_regression",
                    tasks[task_name],
                    task_name,
                    repetition,
                    tasks["inference_pairs"],
                )
                codebert_rows.append(row)
        codebert_status[dataset_name] = {
            "dataset": dataset_name,
            "status": "measured",
            "actual_measurements_available": True,
            "backend": energy_backend,
            "training_repetitions": CODEBERT_TRAINING_REPETITIONS,
            "inference_repetitions": CODEBERT_INFERENCE_REPETITIONS,
            "inference_pairs_per_repetition": tasks["inference_pairs"],
            "pretrained_model_load_included": False,
            "training_includes_embedding_extraction": True,
        }
        torch_module = tasks["torch"]
        del task, tasks
        gc.collect()
        if torch_module.cuda.is_available():
            torch_module.cuda.empty_cache()
elif RUN_CODEBERT_MEASUREMENTS:
    for dataset_name in DATASET_NAMES:
        codebert_status[dataset_name] = {
            "dataset": dataset_name,
            "status": "unavailable",
            "actual_measurements_available": False,
            "reason": "No RAPL or Scaphandre energy counter was detected.",
        }
else:
    for dataset_name in DATASET_NAMES:
        codebert_status[dataset_name] = {
            "dataset": dataset_name,
            "status": "not_run",
            "actual_measurements_available": False,
            "reason": "Set RUN_CODEBERT_MEASUREMENTS = True to opt in.",
        }

codebert_measurements = pd.DataFrame(codebert_rows, columns=measurement_columns)
for dataset_name in DATASET_NAMES:
    output_dir = RESULTS_DIR / dataset_name
    output_dir.mkdir(parents=True, exist_ok=True)
    dataset_measurements = codebert_measurements.loc[codebert_measurements["dataset"].eq(dataset_name)]
    dataset_measurements.to_csv(output_dir / "codebert_energy_measurements.csv", index=False)
    summary = dataset_measurements.groupby(["model", "task", "backend"], as_index=False).agg(
        repetitions=("net_energy_joules", "count"),
        mean_net_energy_joules=("net_energy_joules", "mean"),
        std_net_energy_joules=("net_energy_joules", "std"),
        inference_pairs=("n_inference_pairs", "mean"),
    )
    summary.to_csv(output_dir / "codebert_energy_measurement_summary.csv", index=False)
    (output_dir / "codebert_energy_measurement_status.json").write_text(
        json.dumps(codebert_status[dataset_name], indent=2), encoding="utf-8"
    )
    print(json.dumps(codebert_status[dataset_name], indent=2))
